# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 54), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.86 MiB | 5.91 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank-ml-internship


In [3]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB + HF token ready")

DuckDB + HF token ready


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# **Method:**

Random Forest Classifier (from scikit-learn).

Why it fits: Random Forest handles non-linear relationships between features
(e.g. the age/position interaction we saw in Signal 1 and Signal 2 — where
staleness and volume don't behave as a simple straight line) without needing
manual feature engineering for interactions. It's also robust to outliers
like the low-impression noisy pages we found in ML-07's top-10 review, since
it splits on thresholds rather than relying on raw distances or scale. It
gives feature importances, useful for the Interpretation section, and it's
a standard, well-understood baseline-beating choice for tabular data —
appropriate given this is a first capstone model, not a research paper on
model architecture.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# **Split design:**

 Grouped split by content_hash_id (using sklearn's GroupShuffleSplit),
80/20 train/test — NOT a random row split and NOT time-based.

Why: Each content page (content_hash_id) can have multiple rows if joined
across dimensions, and even with one row per page, using a random split risks
optimistic bias if similar pages cluster in ways the model could memorize.
More importantly, a time-aware split isn't possible here — we only have one
month (March 2026) of warehouse data, so there's no clean future window to
hold out. Grouping by content_hash_id ensures no single page's data leaks
between train and test, which is the honest split given our single-month
data constraint.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Train + compare vs my baseline

# Section 3: Train + compare vs my baseline

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Load the anonymized CSV (has trend_direction + features) ---
model_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
model_df["is_declining"] = (model_df["trend_direction"] == "down").astype(int)

print(f"Total rows: {len(model_df)}")
print(f"Base rate (is_declining=1): {model_df['is_declining'].mean():.3f}")
print(f"Columns available: {model_df.columns.tolist()}")

Total rows: 30000
Base rate (is_declining=1): 0.542
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining']


In [8]:
# Section 3: Train + compare vs my baseline

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# label already exists
model_df["is_declining"] = (model_df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "impressions_last_30d", "avg_position", "ctr"]
model_df = model_df.dropna(subset=features + ["is_declining"])

# --- Grouped split by content_id ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["content_id"]))

train_df = model_df.iloc[train_idx]
test_df = model_df.iloc[test_idx]

X_train, y_train = train_df[features], train_df["is_declining"]
X_test, y_test = test_df[features], test_df["is_declining"]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# --- Baseline: reuse ML-07 rule-based action_score ---
def age_score_fn(days):
    if days < 90: return 0.0
    elif days < 365: return 0.5
    else: return 1.0

test_df = test_df.copy()
test_df["age_score"] = test_df["content_age_days"].apply(age_score_fn)
max_position = train_df["avg_position"].max()
test_df["position_gap_score"] = ((test_df["avg_position"] - 1) / max_position).clip(0, 1)
test_df["baseline_score"] = 0.6 * test_df["age_score"] + 0.4 * test_df["position_gap_score"]

baseline_auc = roc_auc_score(y_test, test_df["baseline_score"])

# --- Model: Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
model_probs = rf.predict_proba(X_test)[:, 1]
model_auc = roc_auc_score(y_test, model_probs)

# --- Comparison table ---
comparison = pd.DataFrame({
    "Method": ["Baseline (rule-based)", "Random Forest"],
    "AUC": [baseline_auc, model_auc],
    "Base rate (is_declining=1)": [model_df["is_declining"].mean()] * 2
})
print(comparison)

Train size: 24000, Test size: 6000
                  Method       AUC  Base rate (is_declining=1)
0  Baseline (rule-based)  0.494226                    0.542067
1          Random Forest  0.759660                    0.542067


**Result:**
Baseline (rule-based score) achieves AUC 0.494 on the held-out
grouped test set — essentially no better than random guessing for this
label. Random Forest achieves AUC 0.760 — a large, genuine improvement,
well above the base rate (0.542), confirming the model captures real
signal the hand-written rule misses.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Errors and interpretation

# --- Feature importances ---
importances = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature importances:")
print(importances)

# --- Error analysis: false positives and false negatives ---
test_df["model_prob"] = model_probs
test_df["model_pred"] = (test_df["model_prob"] > 0.5).astype(int)
test_df["actual"] = y_test.values

false_positives = test_df[(test_df["model_pred"] == 1) & (test_df["actual"] == 0)]
false_negatives = test_df[(test_df["model_pred"] == 0) & (test_df["actual"] == 1)]

print(f"\nFalse positives (predicted declining, actually not): {len(false_positives)}")
print(f"False negatives (predicted not declining, actually declining): {len(false_negatives)}")

print("\nFalse positives — feature averages:")
print(false_positives[features].mean())

print("\nFalse negatives — feature averages:")
print(false_negatives[features].mean())

print("\nCorrect predictions — feature averages:")
correct = test_df[test_df["model_pred"] == test_df["actual"]]
print(correct[features].mean())

Feature importances:
                feature  importance
2          avg_position    0.375394
0      content_age_days    0.257955
1  impressions_last_30d    0.255513
3                   ctr    0.111138

False positives (predicted declining, actually not): 1050
False negatives (predicted not declining, actually declining): 725

False positives — feature averages:
content_age_days        205.660000
impressions_last_30d    485.256190
avg_position             15.131905
ctr                       0.222010
dtype: float64

False negatives — feature averages:
content_age_days         349.040000
impressions_last_30d    2889.401379
avg_position              17.841103
ctr                        0.587103
dtype: float64

Correct predictions — feature averages:
content_age_days         253.005444
impressions_last_30d    1399.369231
avg_position              15.863550
ctr                        0.531759
dtype: float64


# **Feature importances:**
avg_position is the strongest signal (0.375), followed
by content_age_days (0.258) and impressions_last_30d (0.256), with ctr
contributing least (0.111). This confirms position and staleness both matter,
but position matters more than raw age alone — something the fixed baseline
rule (which only combined these two linearly) failed to capture well.

Error analysis:
- False positives (1,050 pages): model predicted decline but they were stable.
  These pages had lower-than-average impressions (485 vs 1,399 correct avg)
  and slightly newer age (206 days) — the model seems to over-flag younger,
  lower-traffic pages as declining, possibly because low-traffic pages are
  noisier and harder to read correctly.

- False negatives (725 pages): model missed real declines. These pages had
  very high impressions (2,889, double the correct-prediction average) —
  the model appears to under-flag high-traffic pages as declining, perhaps
  because high impression volume usually correlates with stronger pages in
  training, so the model leans toward trusting popularity as a stability signal
  even when it's wrong.

Takeaway: the model leans most heavily on position and volume together —
it's strong at catching typical decline patterns but can miss high-traffic
pages that are quietly declining, and can over-flag newer low-traffic pages.
This is a real limitation worth flagging in the capstone's Limitations section.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.